# Interactive Cycle Time Analysis

This notebook provides an interactive interface for analyzing JIRA cycle times with clickable scatter plots that link directly to JIRA issues.

## Features
- Interactive credential input for JIRA connection
- Custom JQL query input
- Clickable scatter plot with JIRA issue links
- Hover tooltips with issue details
- Configurable workflow mapping

In [ ]:
# Import required libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import getpass
import datetime
from jira import JIRA
import logging

# Import project modules
from jira_agile_metrics.fixed_querymanager import create_fixed_query_manager
from jira_agile_metrics.calculators.cycletime import calculate_cycle_times
from jira_agile_metrics.calculators.scatterplot import calculate_scatterplot_data

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✅ Libraries imported successfully")

## 1. Configuration File Setup

Load configuration from a YAML file (same format as the main application):

In [ ]:
# Configuration File Loading
print("📁 Configuration File Setup")
print("=" * 50)

# Import config module
from jira_agile_metrics.config import config_to_options
import os

# Create file selector widget
config_file_input = widgets.Text(
    value='examples/interactive_config_template.yml',
    placeholder='Enter path to config file',
    description='Config File:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

load_config_button = widgets.Button(
    description='Load Configuration',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

config_status = widgets.HTML(value="")
config_details = widgets.HTML(value="")

# Display widgets
display(config_file_input, load_config_button, config_status, config_details)

# Global variables
jira_client = None
query_manager = None
config_options = None

def load_configuration(b):
    global jira_client, query_manager, config_options
    
    try:
        config_status.value = "🔄 Loading configuration..."
        
        config_file = config_file_input.value
        
        # Check if file exists
        if not os.path.exists(config_file):
            config_status.value = f"❌ Config file not found: {config_file}"
            return
        
        # Load and parse configuration
        with open(config_file, 'r') as f:
            config_data = f.read()
        
        config_options = config_to_options(config_data, cwd=os.path.dirname(config_file))
        
        # Create JIRA connection
        connection_config = config_options['connection']
        
        # Determine authentication method
        if connection_config.get('token'):
            auth = (connection_config['username'], connection_config['token'])
        elif connection_config.get('password'):
            auth = (connection_config['username'], connection_config['password'])
        else:
            config_status.value = "❌ No authentication credentials found in config"
            return
        
        # Configure JIRA client options with API v3 and search/jql endpoint fix
        jira_options = {
            'server': connection_config['domain'],
            'rest_api_version': '3'
        }
        
        # Merge with any additional client options from config
        jira_options.update(connection_config.get('jira_client_options', {}))
        
        jira_client = JIRA(
            options=jira_options,
            basic_auth=auth
        )
        
        # Test connection
        user = jira_client.current_user()
        
        # Create QueryManager with loaded settings and fixed endpoint
        query_manager = create_fixed_query_manager(jira_client, config_options['settings'])
        
        config_status.value = f"✅ Configuration loaded and connected as {user}"
        
        # Display configuration details
        queries = config_options['settings']['queries']
        cycle = config_options['settings']['cycle']
        
        details_html = f"""
        <div style="background-color: #f0f0f0; padding: 10px; margin-top: 10px; border-radius: 5px;">
        <h4>Configuration Details:</h4>
        <p><strong>Server:</strong> {connection_config['domain']}</p>
        <p><strong>User:</strong> {connection_config['username']}</p>
        <p><strong>Queries:</strong> {len(queries)} query(ies) configured</p>
        <p><strong>Workflow Stages:</strong> {len(cycle)} stages ({' → '.join([stage['name'] for stage in cycle])})</p>
        <p><strong>Committed Column:</strong> {config_options['settings'].get('committed_column', 'Auto-detected')}</p>
        <p><strong>Done Column:</strong> {config_options['settings'].get('done_column', 'Auto-detected')}</p>
        </div>
        """
        config_details.value = details_html
        
    except Exception as e:
        config_status.value = f"❌ Configuration failed: {str(e)}"
        config_details.value = ""
        jira_client = None
        query_manager = None
        config_options = None

load_config_button.on_click(load_configuration)

# Show sample config file location
print("\n💡 Sample configuration files:")
print("- examples/interactive_config_template.yml (simple template)")
print("- examples/interactive_config.yml (detailed example)")
print("\nCreate your own config file based on these templates.")

## 2. Configuration Review

Review and optionally modify the loaded configuration:

In [ ]:
# Configuration Review and Override
print("⚙️ Configuration Review")
print("=" * 50)
print("Review the configuration loaded from the YAML file.")
print("You can override settings here if needed.")
print()

# Override widgets (initially hidden)
override_workflow_checkbox = widgets.Checkbox(
    value=False,
    description='Override workflow configuration',
    style={'description_width': 'initial'}
)

workflow_text = widgets.Textarea(
    value='',
    placeholder='Enter workflow configuration as Python list',
    description='Workflow:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='800px', height='150px', display='none')
)

committed_column_input = widgets.Text(
    value='',
    description='Committed Stage:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px', display='none')
)

done_column_input = widgets.Text(
    value='',
    description='Done Stage:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px', display='none')
)

override_query_checkbox = widgets.Checkbox(
    value=False,
    description='Override JQL query',
    style={'description_width': 'initial'}
)

jql_override_input = widgets.Textarea(
    value='',
    placeholder='Enter JQL query to override config file',
    description='JQL Query:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='800px', height='100px', display='none')
)

max_results_input = widgets.IntText(
    value=100,
    description='Max Results:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='200px')
)

def on_workflow_override_change(change):
    if change['new']:
        workflow_text.layout.display = 'block'
        committed_column_input.layout.display = 'block'
        done_column_input.layout.display = 'block'
        if config_options:
            workflow_text.value = str(config_options['settings']['cycle'])
            committed_column_input.value = config_options['settings'].get('committed_column', '')
            done_column_input.value = config_options['settings'].get('done_column', '')
    else:
        workflow_text.layout.display = 'none'
        committed_column_input.layout.display = 'none'
        done_column_input.layout.display = 'none'

def on_query_override_change(change):
    if change['new']:
        jql_override_input.layout.display = 'block'
        if config_options and config_options['settings']['queries']:
            jql_override_input.value = config_options['settings']['queries'][0]['jql']
    else:
        jql_override_input.layout.display = 'none'

override_workflow_checkbox.observe(on_workflow_override_change, names='value')
override_query_checkbox.observe(on_query_override_change, names='value')

display(override_workflow_checkbox)
display(workflow_text)
display(widgets.HBox([committed_column_input, done_column_input]))
display(override_query_checkbox)
display(jql_override_input)
display(max_results_input)

## 3. Run Analysis

Execute the cycle time analysis using the loaded configuration:

In [ ]:
# Analysis Execution
analyze_button = widgets.Button(
    description='Run Analysis',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

analysis_status = widgets.HTML(value="")

print("📊 Analysis Execution")
print("=" * 50)
print("Run the cycle time analysis using the configuration from the YAML file.")
print()

display(analyze_button)
display(analysis_status)

# Global variables for analysis results
cycle_data = None
scatter_data = None

def analyze_issues(b):
    global cycle_data, scatter_data
    
    if not jira_client or not config_options:
        analysis_status.value = "❌ Please load configuration first"
        return
    
    try:
        analysis_status.value = "🔄 Fetching and analyzing issues..."
        
        # Get configuration settings
        settings = config_options['settings'].copy()
        
        # Apply overrides if specified
        if override_workflow_checkbox.value:
            settings['cycle'] = eval(workflow_text.value)
            if committed_column_input.value:
                settings['committed_column'] = committed_column_input.value
            if done_column_input.value:
                settings['done_column'] = done_column_input.value
        
        if override_query_checkbox.value and jql_override_input.value:
            settings['queries'] = [{'jql': jql_override_input.value, 'value': 'Analysis'}]
        
        # Apply max results limit
        settings['max_results'] = max_results_input.value
        
        # Update query manager settings
        query_manager.settings.update(settings)
        
        # Auto-detect committed and done columns if not specified
        cycle = settings['cycle']
        committed_column = settings.get('committed_column')
        done_column = settings.get('done_column')
        
        if not committed_column:
            committed_column = cycle[1]['name'] if len(cycle) > 1 else cycle[0]['name']
        if not done_column:
            done_column = cycle[-1]['name']
        
        # Calculate cycle times
        cycle_data = calculate_cycle_times(
            query_manager,
            cycle,
            settings.get('attributes', {}),
            committed_column,
            done_column,
            settings['queries'],
            settings.get('query_attribute')
        )
        
        # Calculate scatter plot data
        scatter_data = calculate_scatterplot_data(cycle_data)
        
        analysis_status.value = f"""✅ Analysis complete! 
        <br>• Found {len(cycle_data)} total issues
        <br>• {len(scatter_data)} issues have cycle times
        <br>• Cycle time calculated from '{committed_column}' to '{done_column}'
        <br>• Using {len(settings['queries'])} quer{'y' if len(settings['queries']) == 1 else 'ies'}
        """
        
    except Exception as e:
        import traceback
        analysis_status.value = f"❌ Analysis failed: {str(e)}<br><pre>{traceback.format_exc()}</pre>"
        cycle_data = None
        scatter_data = None

analyze_button.on_click(analyze_issues)

## 4. Interactive Cycle Time Scatter Plot

Generate an interactive scatter plot with clickable data points:

In [ ]:
def create_interactive_scatter_plot():
    if scatter_data is None or len(scatter_data) == 0:
        print("❌ No data available. Please run the analysis first.")
        return
    
    # Prepare data for plotting
    plot_data = scatter_data.copy()
    
    # Convert cycle_time to days (numeric)
    plot_data['cycle_time_days'] = plot_data['cycle_time'].dt.total_seconds() / (24 * 3600)
    
    # Create hover text with issue details
    plot_data['hover_text'] = (
        "<b>" + plot_data['key'] + "</b><br>" +
        "Summary: " + plot_data['summary'].str[:50] + "...<br>" +
        "Cycle Time: " + plot_data['cycle_time_days'].round(1).astype(str) + " days<br>" +
        "Completed: " + plot_data['completed_date'].dt.strftime('%Y-%m-%d') + "<br>" +
        "Status: " + plot_data['status'] + "<br>" +
        "Type: " + plot_data['issue_type'] + "<br>" +
        "Blocked Days: " + plot_data['blocked_days'].astype(str) + "<br>" +
        "<i>Click to open in JIRA</i>"
    )
    
    # Create the scatter plot
    fig = go.Figure()
    
    # Add scatter trace
    fig.add_trace(go.Scatter(
        x=plot_data['completed_date'],
        y=plot_data['cycle_time_days'],
        mode='markers',
        marker=dict(
            size=8,
            color=plot_data['blocked_days'],
            colorscale='Reds',
            colorbar=dict(title="Blocked Days"),
            line=dict(width=1, color='DarkSlateGrey')
        ),
        text=plot_data['hover_text'],
        hovertemplate='%{text}<extra></extra>',
        customdata=plot_data['url'],
        name='Issues'
    ))
    
    # Add percentile lines
    percentiles = [0.5, 0.85, 0.95]
    colors = ['green', 'orange', 'red']
    
    for i, p in enumerate(percentiles):
        percentile_value = plot_data['cycle_time_days'].quantile(p)
        fig.add_hline(
            y=percentile_value,
            line_dash="dash",
            line_color=colors[i],
            annotation_text=f"{int(p*100)}% ({percentile_value:.1f} days)",
            annotation_position="top left"
        )
    
    # Update layout
    fig.update_layout(
        title={
            'text': 'Interactive Cycle Time Scatter Plot<br><sub>Click on data points to open JIRA issues</sub>',
            'x': 0.5,
            'xanchor': 'center'
        },
        xaxis_title='Completion Date',
        yaxis_title='Cycle Time (Days)',
        hovermode='closest',
        height=600,
        showlegend=False
    )
    
    # Add JavaScript for click handling
    fig.update_layout(
        updatemenus=[
            dict(
                type="buttons",
                direction="left",
                buttons=list([
                    dict(
                        args=[{"visible": [True]}],
                        label="Refresh",
                        method="restyle"
                    )
                ]),
                pad={"r": 10, "t": 10},
                showactive=True,
                x=0.01,
                xanchor="left",
                y=1.02,
                yanchor="top"
            ),
        ]
    )
    
    # Display the plot
    fig.show()
    
    # Add JavaScript for click handling
    display(HTML("""
    <script>
    // Add click handler for opening JIRA links
    document.addEventListener('DOMContentLoaded', function() {
        // Find all plotly graphs
        var graphs = document.querySelectorAll('.plotly-graph-div');
        
        graphs.forEach(function(graph) {
            graph.on('plotly_click', function(data) {
                if (data.points && data.points.length > 0) {
                    var point = data.points[0];
                    if (point.customdata) {
                        window.open(point.customdata, '_blank');
                    }
                }
            });
        });
    });
    </script>
    """))
    
    # Display summary statistics
    print("\n📈 Summary Statistics")
    print("=" * 50)
    print(f"Total Issues: {len(plot_data)}")
    print(f"Average Cycle Time: {plot_data['cycle_time_days'].mean():.1f} days")
    print(f"Median Cycle Time: {plot_data['cycle_time_days'].median():.1f} days")
    print(f"85th Percentile: {plot_data['cycle_time_days'].quantile(0.85):.1f} days")
    print(f"95th Percentile: {plot_data['cycle_time_days'].quantile(0.95):.1f} days")
    print(f"Average Blocked Days: {plot_data['blocked_days'].mean():.1f} days")

# Button to generate the plot
plot_button = widgets.Button(
    description='Generate Interactive Plot',
    button_style='info',
    layout=widgets.Layout(width='250px')
)

def generate_plot(b):
    create_interactive_scatter_plot()

plot_button.on_click(generate_plot)

print("📊 Interactive Visualization")
print("=" * 50)
print("Click the button below to generate an interactive scatter plot.")
print("Data points are colored by blocked days and clickable to open JIRA issues.")
print()

display(plot_button)

## 5. Data Export

Export your analysis results for further processing:

In [ ]:
def export_data():
    if cycle_data is None:
        print("❌ No data available. Please run the analysis first.")
        return
    
    # Export cycle data
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Full cycle data
    cycle_filename = f"cycle_time_data_{timestamp}.csv"
    cycle_data.to_csv(cycle_filename, index=False)
    print(f"✅ Full cycle data exported to: {cycle_filename}")
    
    # Scatter plot data
    if scatter_data is not None:
        scatter_filename = f"scatter_plot_data_{timestamp}.csv"
        scatter_data.to_csv(scatter_filename, index=False)
        print(f"✅ Scatter plot data exported to: {scatter_filename}")
    
    # Summary statistics
    if scatter_data is not None and len(scatter_data) > 0:
        cycle_time_days = scatter_data['cycle_time'].dt.total_seconds() / (24 * 3600)
        summary_stats = {
            'Total Issues': len(scatter_data),
            'Average Cycle Time (days)': cycle_time_days.mean(),
            'Median Cycle Time (days)': cycle_time_days.median(),
            '85th Percentile (days)': cycle_time_days.quantile(0.85),
            '95th Percentile (days)': cycle_time_days.quantile(0.95),
            'Average Blocked Days': scatter_data['blocked_days'].mean()
        }
        
        summary_df = pd.DataFrame(list(summary_stats.items()), columns=['Metric', 'Value'])
        summary_filename = f"summary_statistics_{timestamp}.csv"
        summary_df.to_csv(summary_filename, index=False)
        print(f"✅ Summary statistics exported to: {summary_filename}")

export_button = widgets.Button(
    description='Export Data',
    button_style='warning',
    layout=widgets.Layout(width='150px')
)

export_button.on_click(lambda b: export_data())

print("💾 Data Export")
print("=" * 50)
print("Export your analysis results to CSV files for further processing.")
print()

display(export_button)

## Usage Instructions

1. **Create Config File**: Copy and customize `examples/interactive_config_template.yml`
2. **Load Configuration**: Enter the path to your config file and click "Load Configuration"
3. **Review Settings**: Optionally override workflow or query settings
4. **Run Analysis**: Click "Run Analysis" to fetch and process data
5. **Generate Plot**: Create the interactive scatter plot
6. **Interact**: Click on data points to open JIRA issues in new tabs
7. **Export**: Save results to CSV files for further analysis

## Configuration File Format

The configuration file uses the same YAML format as the main jira-agile-metrics application:

```yaml
connection:
  domain: https://your-company.atlassian.net
  username: your-email@company.com
  token: your-api-token

query: project = "MYPROJ" AND status = Done

workflow:
  Backlog: [Backlog, To Do]
  In Progress: [In Progress, Development]
  Done: [Done, Closed]
```

## Features

- **Configuration-Driven**: Uses same config format as main application
- **Interactive Scatter Plot**: Hover for details, click to open JIRA issues
- **Color Coding**: Points colored by blocked days (red = more blocked time)
- **Percentile Lines**: 50th, 85th, and 95th percentile indicators
- **Summary Statistics**: Key metrics displayed below the plot
- **Data Export**: Export raw data and statistics to CSV

## Tips

- Use API tokens instead of passwords for better security
- Start with a small dataset (set max results) to test your configuration
- Copy the template config file and customize for your environment
- Use the same config file format as your main jira-agile-metrics setup